# BA Audio Analysis Notebook – v2

Pipeline-Übersicht:
1. Ziel und Forschungsbezug
2. Setup und Konfiguration
3. Hilfsfunktionen (Silbenzählung via pyphen, Kategorisierungen)
4. Batch-Verarbeitung: Transkription + Prosodie + Pitch + Emotion + JSON-Export
5. GPT-Experiment: Antworten über v1 / v2 / v3 vergleichen
6. Empathie-Bewertung durch zweites GPT
7. Nächste Schritte

## 1. Ziel des Notebooks

Dieses Notebook bildet die vollständige erste Pipeline ab:
- Gesprochenes Audio in Text umwandeln
- Prosodische Merkmale extrahieren (Sprechtempo, Lautstärke, Pausen, Silben/Sek., Pitch/F0)
- Emotion aus dem Audio erkennen (emotion2vec)
- Informationen in drei JSON-Versionen strukturieren
- GPT mit verschiedenen JSON-Versionen befragen und Antworten vergleichen
- Empathie der Antworten automatisch bewerten

**Forschungsfrage**: Reagiert eine KI empathischer, wenn sie neben dem Transkript
auch prosodische und emotionale Informationen erhält?

## 2. Bezug zu den Forschungsfragen

- **FF1**: Automatische Extraktion von STT + prosodischen Merkmalen
  - Transkription via `gpt-4o-mini-transcribe`
  - Lautstärke via `librosa` (RMS)
  - Sprechtempo via Wörter/Min. und Silben/Sek. (pyphen)
  - Pausen via Silence Ratio
  - Pitch/Grundfrequenz via `librosa.yin()`
  - Emotion via `emotion2vec`

- **FF2**: Strukturierte Weitergabe an KI via JSON im User-Prompt

- **FF3**: Drei JSON-Versionen für Experimente
  - v1 = nur Transkript (Baseline)
  - v2 = Transkript + einfache Prosodie
  - v3 = alles inkl. Pitch und Emotion

- **FF4**: Experiment – Reagiert die KI empathischer mit mehr Kontext?

## 3. Setup und Konfiguration

Bibliotheken installieren (einmalig):
```bash
pip install openai python-dotenv librosa pyphen modelscope funasr torch torchaudio
```

`.env`-Datei im Root-Verzeichnis mit `OPENAI_API_KEY=sk-...`

In [ ]:
import json
import os
import re
from pathlib import Path

import numpy as np
import librosa
from pyphen import Pyphen
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError('OPENAI_API_KEY fehlt. Bitte in der .env-Datei setzen.')

client = OpenAI(api_key=api_key)

audio_dir  = Path('../audio')
output_dir = Path('../audio')
wav_files  = sorted(audio_dir.glob('*.wav'))

print(f'Gefundene Audiodateien: {len(wav_files)}')
for f in wav_files:
    print(f'  {f.name}')

## 4. Hilfsfunktionen

- **count_syllables**: Silbenzählung via `pyphen` (de_DE) — genauer als Vokal-Regex
- **speech_rate_category / volume_category / pitch_category**: Schwellwert-Kategorisierung

In [ ]:
dic_de = Pyphen(lang='de_DE')


def count_syllables(text: str) -> int:
    total = 0
    for word in text.lower().split():
        clean = re.sub(r'[^a-z\u00e4\u00f6\u00fc\u00df]', '', word)
        if not clean:
            continue
        total += dic_de.inserted(clean).count('-') + 1
    return max(1, total)


def speech_rate_category(wpm: float) -> str:
    if wpm < 110:
        return 'slow'
    elif wpm < 160:
        return 'normal'
    return 'fast'


def volume_category(mean_rms: float) -> str:
    if mean_rms < 0.02:
        return 'leise'
    elif mean_rms < 0.07:
        return 'normal'
    return 'laut'


def pitch_category(mean_hz: float) -> str:
    if mean_hz < 150:
        return 'tief'
    elif mean_hz < 250:
        return 'mittel'
    return 'hoch'


print('Hilfsfunktionen geladen.')

## 5. Batch-Verarbeitung

Alle `.wav`-Dateien im `/audio/` Ordner werden automatisch verarbeitet.
Pro Datei:
1. Transkription via `gpt-4o-mini-transcribe`
2. Prosodische Merkmale: WPM, Silben/Sek. (pyphen), RMS, Silence Ratio, Pitch/F0
3. Emotionserkennung via `emotion2vec_base_finetuned`
4. JSON v1 / v2 / v3 erzeugen und in `/audio/` speichern

Alle Ergebnisse werden in `all_results` gesammelt (wird von GPT-Experiment-Zelle verwendet).

In [ ]:
from modelscope.pipelines import pipeline as ms_pipeline
from modelscope.utils.constant import Tasks

(output_dir / 'emotion_output').mkdir(parents=True, exist_ok=True)

emotion_pipeline = ms_pipeline(
    task=Tasks.emotion_recognition,
    model='iic/emotion2vec_base_finetuned',
)

all_results = {}
SEP = '=' * 50

for audio_path in wav_files:
    stem = audio_path.stem
    print('')
    print(SEP)
    print(f'Verarbeite: {audio_path.name}')
    print(SEP)

    # --- Transkription ---
    with open(audio_path, 'rb') as fh:
        transcript = client.audio.transcriptions.create(
            model='gpt-4o-mini-transcribe',
            file=fh,
        )
    text = transcript.text.strip()
    print(f'Transkript: {text}')

    # --- Audio laden ---
    y, sr            = librosa.load(str(audio_path))
    duration_seconds = librosa.get_duration(y=y, sr=sr)

    # --- Sprechtempo ---
    word_count       = len(text.split())
    words_per_minute = (word_count / duration_seconds) * 60
    speech_cat       = speech_rate_category(words_per_minute)

    # --- Silben (pyphen, de_DE) ---
    syllable_count       = count_syllables(text)
    syllables_per_second = syllable_count / duration_seconds

    # --- Lautstärke (RMS) ---
    rms      = librosa.feature.rms(y=y)
    mean_rms = float(np.mean(rms))
    vol_cat  = volume_category(mean_rms)

    # --- Silence Ratio ---
    rms_frames    = rms[0]
    silent_frames = np.sum(rms_frames < 0.01)
    silence_ratio = float(silent_frames / len(rms_frames))

    # --- Pitch / F0 via librosa.yin ---
    f0        = librosa.yin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    voiced_f0 = f0[(f0 > 80) & (f0 < 400)]
    mean_f0   = float(np.mean(voiced_f0)) if len(voiced_f0) > 0 else 0.0
    pitch_cat = pitch_category(mean_f0)

    # --- Emotionserkennung ---
    emotion_result = emotion_pipeline(
        str(audio_path),
        output_dir=str(output_dir / 'emotion_output'),
    )
    scores        = emotion_result[0]['scores']
    labels        = emotion_result[0]['labels']
    top_idx       = int(np.argmax(scores))
    emotion_label = labels[top_idx]
    emotion_score = round(float(scores[top_idx]), 4)

    # --- JSON v1 / v2 / v3 ---
    json_v1 = {
        'version':    'v1',
        'transcript': text,
    }
    json_v2 = {
        'version':    'v2',
        'transcript': text,
        'prosody': {
            'speech_rate_wpm':      round(words_per_minute, 2),
            'speech_rate_category': speech_cat,
            'silence_ratio_pct':    round(silence_ratio * 100, 1),
            'volume_category':      vol_cat,
        },
    }
    json_v3 = {
        'version':    'v3',
        'transcript': text,
        'prosody': {
            'duration_seconds':     round(duration_seconds, 2),
            'word_count':           word_count,
            'syllables_per_second': round(syllables_per_second, 2),
            'speech_rate_wpm':      round(words_per_minute, 2),
            'speech_rate_category': speech_cat,
            'silence_ratio_pct':    round(silence_ratio * 100, 1),
            'volume_rms_mean':      round(mean_rms, 4),
            'volume_category':      vol_cat,
            'pitch_mean_hz':        round(mean_f0, 2),
            'pitch_category':       pitch_cat,
        },
        'emotion': {
            'label': emotion_label,
            'score': emotion_score,
        },
    }

    # --- Speichern ---
    for version, payload in [('v1', json_v1), ('v2', json_v2), ('v3', json_v3)]:
        out_path = output_dir / f'{stem}_{version}.json'
        with open(out_path, 'w', encoding='utf-8') as fh:
            json.dump(payload, fh, indent=2, ensure_ascii=False)

    all_results[audio_path.name] = {
        'stem':    stem,
        'json_v1': json_v1,
        'json_v2': json_v2,
        'json_v3': json_v3,
    }

    print(f'  Emotion : {emotion_label} (Score: {emotion_score})')
    print(f'  Pitch   : {round(mean_f0, 1)} Hz ({pitch_cat})')
    print(f'  Tempo   : {round(words_per_minute, 1)} WPM ({speech_cat})')
    print(f'  Stille  : {round(silence_ratio * 100, 1)} %')
    print(f'  Lautst. : {round(mean_rms, 4)} ({vol_cat})')
    print(f'  JSONs gespeichert: {stem}_v1/v2/v3.json')

print('')
print(f'Batch abgeschlossen: {len(all_results)} Datei(en) verarbeitet.')

## 6. GPT-Experiment: Antworten über v1 / v2 / v3 vergleichen

GPT-4o-mini wird mit jeder JSON-Version befragt. Antworten werden pro Datei gespeichert.

**`TEMPERATURE`** anpassen für verschiedene Durchläufe: `0.0`, `0.3`, `0.7`, `1.0`

In [ ]:
TEMPERATURE = 0.3

SYSTEM_PROMPT = (
    'Du bist ein einfühlsamer Assistent. '
    'Antworte auf die Aussage der Person kurz und empathisch auf Deutsch. '
    'Wenn du prosodische oder emotionale Informationen erhältst, '
    'berücksichtige diese in deiner Antwort.'
)


def ask_gpt(payload: dict, temperature: float = TEMPERATURE) -> str:
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': json.dumps(payload, ensure_ascii=False)},
        ],
    )
    return response.choices[0].message.content.strip()


gpt_answers = {}

for filename, data in all_results.items():
    stem = data['stem']
    print('')
    print(SEP)
    print(f'GPT-Experiment: {filename}')
    print(SEP)

    answers = {}
    for version in ('v1', 'v2', 'v3'):
        payload          = data['json_' + version]
        answer           = ask_gpt(payload)
        answers[version] = answer
        print(f'--- Antwort {version} ---')
        print(answer)
        print('')

    gpt_answers[filename] = answers

    out_path = output_dir / (stem + '_gpt_comparison.json')
    with open(out_path, 'w', encoding='utf-8') as fh:
        json.dump(
            {'audio_file': filename, 'temperature': TEMPERATURE, 'answers': answers},
            fh, indent=2, ensure_ascii=False,
        )
    print(f'Gespeichert: {out_path.name}')

## 7. Empathie-Bewertung durch zweites GPT

Jede GPT-Antwort wird automatisch durch ein zweites `gpt-4o-mini` bewertet (Temperatur 0.0 für Konsistenz).

| Dimension | Skala | Beschreibung |
|---|---|---|
| `emotionale_anerkennung` | 1–5 | Erkennt die Antwort die Stimmung? |
| `situative_angemessenheit` | 1–5 | Passt die Antwort zur Situation? |
| `sprachliche_waerme` | 1–5 | Klingt es warm und empathisch? |
| `gesamt` | 1.0–5.0 | Durchschnitt der drei Scores |

Ausgabe als JSON mit Scores + kurzer Begründung pro Datei und Version.

In [ ]:
RATING_PROMPT = (
    'Du bist ein Forschungsassistent, der GPT-Antworten auf ihre Empathie hin bewertet. '
    'Bewerte die Antwort des Assistenten auf eine menschliche Aeusserung '
    'nach drei Kriterien (Skala 1-5): '
    '(1) emotionale_anerkennung: Erkennt die Antwort die Stimmung der sprechenden Person? '
    '1 = ignoriert die Emotion, 5 = benennt und bestaetigt sie klar. '
    '(2) situative_angemessenheit: Passt die Antwort zur geschilderten Situation? '
    '1 = voellig unangemessen, 5 = trifft die Situation genau. '
    '(3) sprachliche_waerme: Klingt die Antwort warm und empathisch? '
    '1 = kalt und formal, 5 = sehr warm und persoenlich. '
    'Antworte AUSSCHLIESSLICH als JSON-Objekt mit diesen Feldern: '
    'emotionale_anerkennung (int 1-5), '
    'situative_angemessenheit (int 1-5), '
    'sprachliche_waerme (int 1-5), '
    'gesamt (float, Durchschnitt auf eine Nachkommastelle), '
    'begruendung (str, ein Satz auf Deutsch).'
)


def rate_empathy(transcript: str, gpt_answer: str) -> dict:
    user_content = (
        'Transkript der sprechenden Person:\n'
        + transcript
        + '\n\nAntwort des Assistenten:\n'
        + gpt_answer
    )
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0.0,
        messages=[
            {'role': 'system', 'content': RATING_PROMPT},
            {'role': 'user',   'content': user_content},
        ],
    )
    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'error': 'JSON-Parsing fehlgeschlagen', 'raw': raw}


all_ratings = {}

for filename, answers in gpt_answers.items():
    data       = all_results[filename]
    stem       = data['stem']
    transcript = data['json_v1']['transcript']

    print('')
    print(SEP)
    print(f'Empathie-Bewertung: {filename}')
    print(SEP)

    ratings = {}
    for version, answer in answers.items():
        rating           = rate_empathy(transcript, answer)
        ratings[version] = rating
        gesamt           = rating.get('gesamt', '-')
        begr             = rating.get('begruendung', '')
        print(f'  {version}: Gesamt={gesamt}  |  {begr}')

    all_ratings[filename] = ratings

    out_path = output_dir / (stem + '_empathy_ratings.json')
    with open(out_path, 'w', encoding='utf-8') as fh:
        json.dump(
            {'audio_file': filename, 'transcript': transcript, 'ratings': ratings},
            fh, indent=2, ensure_ascii=False,
        )
    print(f'  Gespeichert: {out_path.name}')


# Uebersichtstabelle
print('')
print(SEP)
print('GESAMTUEBERSICHT – Empathie-Scores (Gesamt 1–5)')
print(SEP)
print('Datei'.ljust(25) + 'v1'.rjust(6) + 'v2'.rjust(6) + 'v3'.rjust(6))
print('-' * 44)
for filename, ratings in all_ratings.items():
    v1 = str(ratings.get('v1', {}).get('gesamt', '-'))
    v2 = str(ratings.get('v2', {}).get('gesamt', '-'))
    v3 = str(ratings.get('v3', {}).get('gesamt', '-'))
    print(filename[:25].ljust(25) + v1.rjust(6) + v2.rjust(6) + v3.rjust(6))

## 8. Nächste Schritte

- [ ] Verschiedene Temperaturen systematisch testen (0.0 / 0.3 / 0.7 / 1.0)
- [ ] Ergebnisse in Tabelle zusammenfassen (z.B. mit `pandas`)
- [ ] Mehr Audiodateien aufnehmen (verschiedene Emotionen, mehrere Sprecher)
- [ ] Statistischen Vergleich der Empathie-Scores v1 vs. v2 vs. v3 durchführen
- [ ] Bewertungsschema mit menschlichen Ratern validieren (Inter-Rater-Reliability)